In [ ]:
#read files#

import pandas as pd

#Display
pd.set_option("display.max_rows", None)  # 取消列數上限
pd.set_option("display.max_columns", None)  # 如果有很多欄位也取消限制
pd.set_option('display.float_format', '{:.2f}'.format)

#CSV
users_df = pd.read_csv(r'C:\Users\user\OneDrive\Desktop\Chrissy\code\archive\users_data.csv')
transactions_df = pd.read_csv(r'C:\Users\user\OneDrive\Desktop\Chrissy\code\archive\transactions_data.csv')
cards_df = pd.read_csv(r'C:\Users\user\OneDrive\Desktop\Chrissy\code\archive\cards_data.csv')

import json
#json_train_fraud_labels
with open(r'C:\Users\user\OneDrive\Desktop\Chrissy\code\archive\train_fraud_labels.json') as f:
    train_fraud_labels = json.load(f)
if 'target' in train_fraud_labels:
    train_fraud_labels_dict = train_fraud_labels['target']
else:
    train_fraud_labels_dict = train_fraud_labels

train_fraud_labels_df = pd.DataFrame(
    list(train_fraud_labels_dict.items()),
    columns=['transaction_id', 'fraud_label']
)

#json_mcc
with open(r'C:\Users\user\OneDrive\Desktop\Chrissy\code\archive\mcc_codes.json') as f:
    mcc_codes = json.load(f)
mcc_codes_df = pd.DataFrame.from_dict(mcc_codes, orient='index').reset_index()
mcc_codes_df.columns = ['mcc_code', 'category_name']

In [ ]:
#rename columns#

#transactions
transactions_df = transactions_df.rename(columns={'mcc': 'mcc_code'})
transactions_df = transactions_df.rename(columns={'id': 'transaction_id'})

#users
users_df = users_df.rename(columns={'id':'client_id'})

#cards
cards_df = cards_df.rename(columns={'id':'card_id'})

In [ ]:
#missing value checking & missing indicator#

def check_missing(df, df_name, auto_add_flag=True):
    missing_info = df.isnull().sum()
    missing_info = missing_info[missing_info > 0]

    if missing_info.empty:
        print(f"{df_name}: 無缺值")
    else:
        missing_percentage = (missing_info / len(df)) * 100
        missing_df = pd.DataFrame({
            'Missing Count': missing_info,
            'Missing Percentage': missing_percentage
        }).sort_values(by='Missing Count', ascending=False)

        print(f"{df_name}: 有缺值")
        print(missing_df)
        print("-" * 40)

        if auto_add_flag:
            for col in missing_info.index:
                flag_col = f"{col}_missing_flag"
                if flag_col not in df.columns:
                    df[flag_col] = df[col].isna().astype(int)
                    print(f"已新增缺失指標欄位: {flag_col}")


check_missing(transactions_df, "transactions_df")
check_missing(cards_df, "cards_df")
check_missing(train_fraud_labels_df, "train_fraud_labels_df")
check_missing(users_df, "users_df")
check_missing(mcc_codes_df, "mcc_codes_df")

In [ ]:
#missing value imputation#

#merchant_state
transactions_df['merchant_city'] = transactions_df['merchant_city'].astype('category')
transactions_df.loc[
    transactions_df['merchant_city'].str.lower() == 'online', 'merchant_state'
] = 'online'

#zip (20250927)
transactions_df.loc[
    transactions_df['zip'].isna() & (transactions_df['merchant_city'].str.lower() == 'online'),
    'zip'
] = -1

transactions_df.loc[
    transactions_df['zip'].isna() & (transactions_df['merchant_city'].str.lower() != 'online'),
    'zip'
] = -999

#errors
transactions_df['errors'] = transactions_df['errors'].astype('category')
transactions_df['errors'] = transactions_df['errors'].cat.add_categories('No_error').fillna('No_error')

In [ ]:
#format conversion#

import pandas as pd
import json

#transactions
transactions_df['date'] = pd.to_datetime(transactions_df['date'])
transactions_df['amount'] = transactions_df['amount'].replace(r'[\$,]', '', regex=True).astype(float).astype(int)
transactions_df['use_chip'] = transactions_df['use_chip'].astype('category')
transactions_df['merchant_state'] = transactions_df['merchant_state'].astype('category')
transactions_df['mcc_code'] = transactions_df['mcc_code'].astype('int64')
transactions_df['zip'] = transactions_df['zip'].astype('int64') #20250927

#user
users_df['per_capita_income'] = [int(i[1:]) for i in users_df['per_capita_income']]
users_df['yearly_income'] = [int(i[1:]) for i in users_df['yearly_income']]
users_df['total_debt'] = [int(i[1:]) for i in users_df['total_debt']]
users_df['gender'] = users_df['gender'].astype('category')

#cards
cards_df['expires'] = pd.to_datetime(cards_df['expires'], format='%m/%Y')
cards_df['acct_open_date'] = pd.to_datetime(cards_df['acct_open_date'], format='%m/%Y')
cards_df['credit_limit']= [int(i[1:]) for i in cards_df['credit_limit']]
cards_df['card_brand'] = cards_df['card_brand'].astype('category')
cards_df['card_type'] = cards_df['card_type'].astype('category')
cards_df['has_chip'] = cards_df['has_chip'].map({'YES': 1, 'NO': 0})
cards_df['has_chip'] = cards_df['has_chip'].astype('int64')

#train_fraud_labels
train_fraud_labels_df['transaction_id'] = train_fraud_labels_df['transaction_id'].astype('int64')
train_fraud_labels_df['fraud_label'] = train_fraud_labels_df['fraud_label'].map({'Yes': True, 'No': False})

#MCC
mcc_codes_df['mcc_code'] = mcc_codes_df['mcc_code'].astype('int64')

In [ ]:
#one hot encoding#

#cards_df
cols_to_encode = ['card_type', 'card_brand']
dummies_cards = pd.get_dummies(cards_df[cols_to_encode], prefix=cols_to_encode, dtype='uint8')
cards_df = pd.concat([cards_df, dummies_cards], axis=1)

#transactions_df
dummies_chip = pd.get_dummies(transactions_df['use_chip'], prefix='use_chip', dtype='uint8')
transactions_df = pd.concat([transactions_df, dummies_chip], axis=1)

In [ ]:
#dataset integration#

#fraud
int_transactions_df = transactions_df.merge(train_fraud_labels_df, on='transaction_id', how='inner')
#int_transactions_df = transactions_df.merge(train_fraud_labels_df, on='transaction_id', how='left')


#users
int_transactions_df = int_transactions_df.merge(users_df, on='client_id', how='left')

#cards
int_transactions_df = int_transactions_df.merge(
    cards_df,
    left_on=['card_id', 'client_id'],
    right_on=['card_id', 'client_id'],
    how='left'
)

check_missing(int_transactions_df, "int_transactions_df")

#MCC(Optional)
#int_transactions_df = int_transactions_df.merge(mcc_codes_df, on='mcc_code', how='left')

#del users_df
#del transactions_df
#del cards_df
#del train_fraud_labels_df
#del mcc_codes_df

In [ ]:
int_transactions_df.shape

In [ ]:
int_transactions_df.info()

In [ ]:
import pandas as pd

# 指定要計算的欄位
cols = [
    'amount', 'current_age', 'retirement_age', 'per_capita_income', 'yearly_income',
    'total_debt', 'credit_score', 'num_credit_cards', 'has_chip', 
    'num_cards_issued', 'credit_limit', 'year_pin_last_changed'
]

# 選取特定欄位的 DataFrame
df_selected = int_transactions_df[cols]

# 生成 describe 統計量
desc = df_selected.describe().T  # 轉置方便閱讀

# 加上 skew 和 kurtosis
desc['skew'] = df_selected.skew()
desc['kurtosis'] = df_selected.kurtosis()

print(desc)


In [ ]:
#EDA_historgram#

import matplotlib.pyplot as plt
import seaborn as sns

hist_df = int_transactions_df.copy()

columns_to_drop = [
    'transaction_id','client_id','card_id','merchant_id',
    'date','acct_open_date','year_pin_last_changed',
    'birth_year','birth_month',
    'zip','merchant_city','merchant_state',
    'errors',
    'fraud_label',
    'address','latitude', 'longitude',
    'expires','cvv',
    'card_on_dark_web',
    'use_chip_Chip Transaction','use_chip_Online Transaction','use_chip_Swipe Transaction',
    'card_type_Credit','card_type_Debit','card_type_Debit (Prepaid)',
    'card_brand_Amex','card_brand_Discover','card_brand_Mastercard','card_brand_Visa',
    'mcc_code',
    'merchant_state_missing_flag','zip_missing_flag','errors_missing_flag','fraud_label_missing_flag',
    'amount','per_capita_income', 'yearly_income', 'total_debt',
    'current_age', 'retirement_age', 
    'credit_score', 'credit_limit',
    'card_number',
]

cate_cols = hist_df.drop(columns=columns_to_drop).columns

for col in cate_cols:
    plt.figure(figsize=(6, 4))
    hist_df[col].value_counts().plot(kind='bar', color='dimgray')
    plt.title(f'Category Distribution for {col}')
    plt.xlabel(col)
    plt.ylabel('Count')
    plt.xticks(rotation=0)

In [ ]:
#EDA_heatmap#
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

corr_df = int_transactions_df.copy()

columns_to_drop = [
    'transaction_id',
    'client_id',
    'card_id',
    'merchant_id',
    'date',
    'use_chip',
    'merchant_city',
    'merchant_state',
    'errors',
    'fraud_label',
    'address',
    'gender',
    'card_brand',
    'card_type',
    'expires',
    'cvv',
    'acct_open_date',
    'card_on_dark_web',
    'use_chip_Chip Transaction',
    'use_chip_Online Transaction',
    'use_chip_Swipe Transaction',
    'card_type_Credit',
    'card_type_Debit',
    'card_type_Debit (Prepaid)',
    'card_brand_Amex',
    'card_brand_Discover',
    'card_brand_Mastercard',
    'card_brand_Visa',
]

corr_df = corr_df.drop(columns=columns_to_drop)

corr = corr_df.corr(method="pearson")

plt.figure(figsize=(15, 10))
sns.heatmap(
    corr,
    annot=True,                # 在格子上顯示數值
    fmt=".1f",                 # 顯示小數點後1位
    cmap="coolwarm",           # 色系 (藍-紅)
    center=0,                  # 中心設在 0 (相關性沒有偏移)
    linewidths=0.5,            # 格子間的線條
    cbar_kws={"shrink": 0.8}   # colorbar 縮小一點
)

plt.title("Correlation Heatmap", fontsize=16)
plt.show()

In [ ]:
#EDA_scatter#

scatter_df = int_transactions_df.copy()

columns_to_drop = [
    'transaction_id','client_id','card_id','merchant_id',
    'birth_month',
    'zip','merchant_city','merchant_state',
    'errors',
    'fraud_label',
    'address','latitude', 'longitude',
    'gender',
    'card_brand','card_type','expires',
    'expires','cvv',
    'date','acct_open_date','year_pin_last_changed',
    'card_on_dark_web',
    'use_chip_Chip Transaction','use_chip_Online Transaction','use_chip_Swipe Transaction',
    'card_type_Credit','card_type_Debit','card_type_Debit (Prepaid)',
    'card_brand_Amex','card_brand_Discover','card_brand_Mastercard','card_brand_Visa',
    'merchant_state_missing_flag','zip_missing_flag','errors_missing_flag',
]

scatter_df = scatter_df.drop(columns=columns_to_drop)

#

import matplotlib.pyplot as plt
from matplotlib.pyplot import subplots

pd.plotting.scatter_matrix(scatter_df, figsize=(22, 22))

plt.show()